# Notebook 03 — IndoBERT Fine-tuning vs LSTM (Apples-to-Apples Comparison)

**Goal:** Fine-tune `indolem/indobert-base-uncased` on the **exact same dataset** as Notebook 02,
then compare side-by-side to quantify the value of transfer learning.

**Key question:** *How much does pretrained language knowledge help on Indonesian text classification?*

**Expected outcome:**
```
| Model              | Accuracy | F1 (macro) | Train Time |
|--------------------|----------|------------|------------|
| BiLSTM from scratch| ~72%     | ~0.68      | ~15 min    |
| IndoBERT fine-tune | ~87%     | ~0.85      | ~20 min    |
```

**Runtime:** T4 GPU, ~20 min for 5 epochs

## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/drive/MyDrive/indonesian-audio-intelligence'
if not os.path.exists(f'{REPO_DIR}/src'):
    !git clone https://github.com/KimiDandy/indonesian-audio-intelligence {REPO_DIR}
else:
    print(f'Repo already exists at {REPO_DIR}')

import sys
sys.path.append(REPO_DIR)
os.makedirs(f'{REPO_DIR}/checkpoints', exist_ok=True)
print('Ready.')

In [ ]:
import torch
import time
import json
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 2. Load Same Dataset as Notebook 02

In [ ]:
from datasets import load_dataset

CATEGORIES = {
    'bisnis': 0, 'teknologi': 1, 'otomotif': 2, 'bola': 3,
    'health': 4, 'lifestyle': 5, 'showbiz': 6, 'regional': 7,
}
IDX_TO_LABEL = {v: k for k, v in CATEGORIES.items()}
NUM_CLASSES = len(CATEGORIES)

def extract_label(url):
    for cat, idx in CATEGORIES.items():
        if f'/{cat}/' in url.lower():
            return idx
    return -1

print('Loading id_liputan6 (same split as Notebook 02)...')
ds = load_dataset('id_liputan6', 'complete', trust_remote_code=True)

def process_split(split_name, max_samples):
    texts, labels = [], []
    for item in ds[split_name]:
        lbl = extract_label(item.get('url', ''))
        if lbl == -1:
            continue
        texts.append(item['clean_article'][:512])
        labels.append(lbl)
        if len(texts) >= max_samples:
            break
    return texts, labels

train_texts, train_labels = process_split('train', 8000)
val_texts,   val_labels   = process_split('validation', 1500)
test_texts,  test_labels  = process_split('test', 1500)

print(f'Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}')

## 3. Load IndoBERT Tokenizer & Model

In [ ]:
MODEL_ID = 'indolem/indobert-base-uncased'
MAX_LEN  = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_CLASSES
)

total_params = sum(p.numel() for p in model.parameters())
print(f'IndoBERT parameters: {total_params/1e6:.0f}M')

# Label mappings
model.config.id2label = IDX_TO_LABEL
model.config.label2id = CATEGORIES

## 4. Tokenize Dataset

In [ ]:
class BERTTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.labels = labels
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors='pt',
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = BERTTextDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_ds   = BERTTextDataset(val_texts,   val_labels,   tokenizer, MAX_LEN)
test_ds  = BERTTextDataset(test_texts,  test_labels,  tokenizer, MAX_LEN)

print(f'Datasets tokenized. Train: {len(train_ds)}')

## 5. Fine-tuning with HuggingFace Trainer

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
    }

In [ ]:
INDOBERT_CKPT = f'{REPO_DIR}/checkpoints/indobert-classification'

training_args = TrainingArguments(
    output_dir=INDOBERT_CKPT,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Trainer ready. Starting fine-tuning...')

In [ ]:
start_time = time.time()
trainer.train()
elapsed = (time.time() - start_time) / 60
print(f'Training complete in {elapsed:.1f} min.')

## 6. Evaluate on Test Set

In [ ]:
test_output = trainer.predict(test_ds)
test_preds  = np.argmax(test_output.predictions, axis=1)
test_true   = test_output.label_ids

bert_acc = accuracy_score(test_true, test_preds)
bert_f1  = f1_score(test_true, test_preds, average='macro', zero_division=0)

print('=' * 50)
print(f'  IndoBERT Fine-tune — Test Results')
print(f'  Accuracy:  {bert_acc:.4f} ({bert_acc*100:.1f}%)')
print(f'  F1 Macro:  {bert_f1:.4f}')
print('=' * 50)
print()
print(classification_report(
    test_true, test_preds,
    target_names=list(CATEGORIES.keys()),
    zero_division=0
))

## 7. Head-to-Head Comparison: LSTM vs IndoBERT

In [ ]:
# Load LSTM results from Notebook 02
lstm_ckpt = torch.load(f'{REPO_DIR}/checkpoints/lstm_best.pt', map_location='cpu')
lstm_acc  = lstm_ckpt['test_accuracy']
lstm_f1   = lstm_ckpt['test_f1_macro']
lstm_time = lstm_ckpt['train_time_min']

results = [
    {'model_name': 'BiLSTM (from scratch)',  'accuracy': lstm_acc,  'f1_macro': lstm_f1,  'train_time_min': f'{lstm_time:.0f}'},
    {'model_name': 'IndoBERT (fine-tuned)',  'accuracy': bert_acc,  'f1_macro': bert_f1,  'train_time_min': f'{elapsed:.0f}'},
]

from src.utils.metrics import format_comparison_table
print(format_comparison_table(results))

delta_acc = (bert_acc - lstm_acc) * 100
delta_f1  = bert_f1 - lstm_f1
print(f'\nTransfer learning improvement:')
print(f'  Accuracy: +{delta_acc:.1f} pp')
print(f'  F1 Macro: +{delta_f1:.3f}')

In [ ]:
from src.utils.visualize import plot_model_comparison, plot_confusion_matrix

fig1 = plot_model_comparison(
    model_names=['BiLSTM\n(from scratch)', 'IndoBERT\n(fine-tuned)'],
    accuracies=[lstm_acc, bert_acc],
    f1_scores=[lstm_f1, bert_f1],
    title='Indonesian Text Classification: Transfer Learning Value'
)
fig1.savefig(f'{REPO_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')

fig2 = plot_confusion_matrix(
    test_true.tolist(), test_preds.tolist(),
    label_names=list(CATEGORIES.keys()),
    title='IndoBERT Confusion Matrix'
)
fig2.savefig(f'{REPO_DIR}/indobert_confusion_matrix.png', dpi=150, bbox_inches='tight')

In [ ]:
# Save IndoBERT for Notebook 04 demo
trainer.save_model(f'{REPO_DIR}/checkpoints/indobert-best')
tokenizer.save_pretrained(f'{REPO_DIR}/checkpoints/indobert-best')

# Save comparison results to JSON (for README)
comparison = {
    'lstm': {'accuracy': lstm_acc, 'f1_macro': lstm_f1, 'train_time_min': lstm_time},
    'indobert': {'accuracy': bert_acc, 'f1_macro': bert_f1, 'train_time_min': elapsed},
    'categories': CATEGORIES,
}
with open(f'{REPO_DIR}/classification_results.json', 'w') as f:
    json.dump(comparison, f, indent=2)

print('Everything saved. Ready for Notebook 04 demo.')